<a href="https://colab.research.google.com/github/GahlasAicha/rag-sql-music-assistant/blob/main/rag_mcp_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb sentence-transformers groq python-dotenv -q

In [ ]:
import chromadb
import sentence_transformers
import groq
print ("toutes les bibliotheques sont installees correctement ")

toutes les bibliotheques sont installees correctement 


In [ ]:
from google.colab import userdata
from groq import Groq # gerer la communication avec les serveurs de Groq
api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key = api_key)
response = client.chat.completions.create(
    model ="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Dis bonjour en une phrase"}]
)
print(response.choices[0].message.content)

Bonjour, je suis ChatGPT, ravi de pouvoir vous aider aujourd’hui.


In [ ]:
!wget -q https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite -O chinook.db
print("Base de donnees telechargee")

Base de donnees telechargee


In [ ]:
import sqlite3

conn = sqlite3.connect("chinook.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Tables disponibles ds la base : ")
for table in tables :
  print(f" - {table[0]}")

conn.close()

Tables disponibles ds la base : 
 - Album
 - Artist
 - Customer
 - Employee
 - Genre
 - Invoice
 - InvoiceLine
 - MediaType
 - Playlist
 - PlaylistTrack
 - Track


In [ ]:
def query_database(sql_query):
  """
  execute une requete SQL sur la base CHinook et renvoie les resultats
  """
  conn = sqlite3.connect("chinook.db")
  cursor =conn.cursor()
  try :
    cursor.execute(sql_query)
    columns = [description[0] for description in cursor.description]
    rows = cursor.fetchall()
    conn.close()
    return {"columns": columns, "rows": rows}
  except Exception as e:
        conn.close()
        return {"error": str(e)}

result = query_database("SELECT Name FROM Artist LIMIT 5;")
print(result)

{'columns': ['Name'], 'rows': [('AC/DC',), ('Accept',), ('Aerosmith',), ('Alanis Morissette',), ('Alice In Chains',)]}


In [ ]:
tools = [
    {
        "type": "function",
        "function":{
            "name": "query_database",
            "description":"Execute une requete SQL en lecture seule sur la base de donnees chinook",
            "parameters":{
                "type": "object",
                "properties":{
                    "sql_query":{
                        "type":"string",
                        "description" : " la requete SQL a executer , en SELECT uniquement "
                    }
                },
                "required": ["sql_query"]
            }
        }
    }
]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":"combien d'artistes y'a til dans la base ?"}],
    tools = tools ,
    tool_choice = "auto"
)
print(response.choices[0].message)

ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='User: French: "combien d\'artistes y\'a til dans la base ?" Means: "How many artists are in the database?" They want count of artists. We need to query the database chinook. Use query_database function. Probably need to select count from Artists table. Let\'s do that.', tool_calls=[ChatCompletionMessageToolCall(id='fc_a8a0e41c-1f79-492a-a8d6-053987fcfe6c', function=Function(arguments='{"sql_query":"SELECT COUNT(*) AS artist_count FROM artists;"}', name='query_database'), type='function')])


In [ ]:
def get_schema():
  conn = sqlite3.connect("chinook.db")
  cursor = conn.cursor()
  cursor.execute("SELECT name FROM sqlite_master WHERE type='table'; ")
  tables = cursor.fetchall()

  schema =""
  for table in tables :
    table_name = table[0]
    cursor.execute(f"PRAGMA table_info({table_name});")
    columns = cursor.fetchall()
    column_names = [col[1] for col in columns]
    schema += f"Table {table_name}: {', '.join(column_names)}\n"

  conn.close()
  return schema
schema_description = get_schema()
print(schema_description)


Table Album: AlbumId, Title, ArtistId
Table Artist: ArtistId, Name
Table Customer: CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId
Table Employee: EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country, PostalCode, Phone, Fax, Email
Table Genre: GenreId, Name
Table Invoice: InvoiceId, CustomerId, InvoiceDate, BillingAddress, BillingCity, BillingState, BillingCountry, BillingPostalCode, Total
Table InvoiceLine: InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity
Table MediaType: MediaTypeId, Name
Table Playlist: PlaylistId, Name
Table PlaylistTrack: PlaylistId, TrackId
Table Track: TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice



In [ ]:
import json

messages = [
    {
    "role" : "system",
    "content" : f"tu est un assisatnt pour une boutique de musique , voici le schema de la base de données disponible :\n{schema_description}\nUtilise l'outil query_database pour repondre aux questions necessitant des donnees reelles. "
    },
    {
        "role":"user","content":"Combient d'artistes y a-t-il dans la base?"}
]
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)
message = response.choices[0].message
if message.tool_calls :
  tool_call = message.tool_calls[0]
  args = json.loads(tool_call.function.arguments)
  print(f"le modele veut executer :{args['sql_query']}")

  result = query_database(args['sql_query'])
  print(f"Resultat de la reponse:{result}")


  messages.append(message)
  messages.append(
     {
         "role":"tool",
         "tool_call_id": tool_call.id,
         "content" : json.dumps(result)
     }
    )
  final_response = client.chat.completions.create(
      model="openai/gpt-oss-20b",
      messages=messages
   )
  print("\nReponse finale du modele :")
  print(final_response.choices[0].message.content)
else:
    print(message.content)



le modele veut executer :SELECT COUNT(*) as artist_count FROM Artist;
Resultat de la reponse:{'columns': ['artist_count'], 'rows': [(275,)]}

Reponse finale du modele :
Il y a **275 artistes** enregistrés dans la base de données.


In [ ]:
import requests
import time

topics = [
    "Rock music", "Jazz", "Heavy metal music", "Blues",
    "Reggae", "Latin music", "Classical music", "Hip hop music"
]

documents = []

headers = {
    "User-Agent": "RAG-MCP-Demo/1.0 (projet etudiant; contact: https://github.com/GahlasAicha)"
}

for topic in topics:
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "prop": "extracts",
            "explaintext": 1,
            "titles": topic,
            "redirects":1,
            "format": "json"
        },
        headers=headers,
        timeout=30
    )
    print(f"{topic} -> statut HTTP {response.status_code}")

    if response.status_code != 200:
        print("  Debut de la reponse :", response.text[:150])
        time.sleep(3)
        continue

    pages = response.json()["query"]["pages"]
    text = list(pages.values())[0].get("extract", "")
    documents.append({"title": topic, "text": text})
    print(f"  {len(text)} caracteres")
    time.sleep(1)

print(f"\n{len(documents)} articles telecharges")

Rock music -> statut HTTP 200
  86790 caracteres
Jazz -> statut HTTP 200
  99508 caracteres
Heavy metal music -> statut HTTP 200
  78673 caracteres
Blues -> statut HTTP 200
  61804 caracteres
Reggae -> statut HTTP 200
  41037 caracteres
Latin music -> statut HTTP 200
  22058 caracteres
Classical music -> statut HTTP 200
  42902 caracteres
Hip hop music -> statut HTTP 200
  50168 caracteres

8 articles telecharges


In [ ]:
def split_into_chunks(text , chunk_size=200 , overlap=40):
  words = text.split()
  chunks= []
  start = 0
  while start < len(words):
    end = start + chunk_size
    chunks.append(" ".join(words[start:end]))
    start += chunk_size - overlap
  return chunks
all_chunks = []
for doc in documents:
    chunks = split_into_chunks(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "id": f"{doc['title']}_{i}",
            "title": doc["title"],
            "text": chunk
        })

print(f"{len(all_chunks)} chunks au total")
print("\nExemple de chunk :")
print("Source :", all_chunks[0]["title"])
print(all_chunks[0]["text"][:300])


483 chunks au total

Exemple de chunk :
Source : Rock music
Rock music is a genre of popular music that originated in the United States as "rock and roll" in the late 1940s and early 1950s, developing into a range of styles from the late 50s to mid-1960s, primarily in the United States and United Kingdom. It has its roots beginning as classic rock and roll, 


In [ ]:
sample = documents[0]["text"]
print("Debut du texte brut :")
print(repr(sample[:300]))
print("\nNombre d'espaces dans l'article :", sample.count(" "))
print("Nombre de caracteres :", len(sample))

Debut du texte brut :
'Rock music is a genre of popular music that originated in the United States as "rock and roll" in the late 1940s and early 1950s, developing into a range of styles from the late 50s to mid-1960s, primarily in the United States and United Kingdom. It has its roots beginning as classic rock and roll, '

Nombre d'espaces dans l'article : 13820
Nombre de caracteres : 86790


In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")


texts = [c["text"] for c in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True)
print("Forme des embeddings :", embeddings.shape)

# Base vectorielle en memoire
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="music_wikipedia")

collection.upsert(
    ids=[c["id"] for c in all_chunks],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"title": c["title"]} for c in all_chunks]
)

print(collection.count(), "chunks stockes dans ChromaDB")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Forme des embeddings : (483, 384)
483 chunks stockes dans ChromaDB


In [ ]:
question = "Where does reggae come from?"
q_emb = model.encode([question]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Source : {meta['title']} (distance : {dist:.2f})")
    print(doc[:200], "...\n")

Source : Reggae (distance : 0.65)
Reggae ( ) is a music genre that originated in Jamaica in the late 1960s. The term also refers to the modern popular music of Jamaica and its diaspora. The 1968 single by Toots and the Maytals titled  ...

Source : Reggae (distance : 0.67)
word we used to use in Jamaica called "streggae". If a girl is walking and the guys look at her and say "Man, she's streggae" it means she don't dress well, she look raggedy. The girls would say that  ...

Source : Reggae (distance : 0.67)
guitar or piano chords on the offbeat. The bass guitar plays a central role, with a thick, heavy tone and the high frequencies reduced to accentuate the low end. Call-and-response patterns are common, ...



In [ ]:
import json


def search_knowledge(query, n_results=3):
    q_emb = model.encode([query]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=n_results)
    passages = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        passages.append({"source": meta["title"], "text": doc})
    return {"passages": passages}


tools = [
    {
        "type": "function",
        "function": {
            "name": "query_database",
            "description": "Execute une requete SQL en lecture seule sur la base Chinook (boutique de musique : artistes, albums, morceaux, clients, factures). A utiliser pour les chiffres reels : ventes, catalogue, clients.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql_query": {"type": "string", "description": "La requete SQL, en SELECT uniquement."}
                },
                "required": ["sql_query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "Recherche dans des articles Wikipedia (en anglais) sur les genres musicaux : rock, jazz, heavy metal, blues, reggae, latin, classique, hip hop. A utiliser pour l'histoire, les origines et les caracteristiques d'un genre.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "La requete de recherche, ecrite en anglais."}
                },
                "required": ["query"]
            }
        }
    }
]


def ask_agent(question, max_steps=5):
    messages = [
        {
            "role": "system",
            "content": f"Tu es un assistant pour une boutique de musique. Reponds en francais. Tu as deux outils : query_database pour les donnees reelles de la boutique, et search_knowledge pour le contexte culturel issu de Wikipedia. Les noms de tables et colonnes sont sensibles a la casse. Quand tu utilises search_knowledge, appuie ta reponse sur les passages retournes.\n\nSchema de la base :\n{schema_description}"
        },
        {"role": "user", "content": question}
    ]
    sources = []

    for _ in range(max_steps):
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        message = response.choices[0].message

        if not message.tool_calls:
            return message.content, list(dict.fromkeys(sources))

        messages.append(message)
        for tool_call in message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            if tool_call.function.name == "query_database":
                result = query_database(args["sql_query"])
                sources.append(f"Base Chinook : {args['sql_query']}")
            elif tool_call.function.name == "search_knowledge":
                result = search_knowledge(args["query"])
                for p in result["passages"]:
                    sources.append(f"Wikipedia : {p['source']}")
            else:
                result = {"error": "outil inconnu"}
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

    return "Je n'ai pas pu terminer la reponse.", list(dict.fromkeys(sources))


questions = [
    "Combien d'albums a le groupe AC/DC ?",
    "D'ou vient le reggae ?",
    "Quel genre est le plus represente dans la boutique, et que sait-on de ce genre ?"
]

for q in questions:
    answer, sources = ask_agent(q)
    print("Question :", q)
    print("Reponse :", answer)
    print("Sources :", sources)
    print("-" * 40)

Question : Combien d'albums a le groupe AC/DC ?
Reponse : Le groupe **AC/DC** possède **2 albums** dans le catalogue de la boutique.
Sources : ["Base Chinook : SELECT COUNT(*) AS album_count FROM Album WHERE ArtistId = (SELECT ArtistId FROM Artist WHERE Name = 'AC/DC');"]
----------------------------------------
Question : D'ou vient le reggae ?
Reponse : Le reggae est né en Jamaïque à la fin des années 1960. Il a émergé comme évolution de styles jamaïcains plus rapides, notamment le **ska** et le **rocksteady**, tout en intégrant des influences du **mento** (une musique folklorique rurale) et des rythmes africains traditionnels.  

> *"Reggae ( ) is a music genre that originated in Jamaica in the late 1960s… Reggae developed from earlier Jamaican genres including mento, ska, and rocksteady, and is rooted in traditional drumming styles such as Kumina, Pukkumina, Revival Zion, Nyabinghi, and burru."*  

Le nom même du genre vient du titre **“Do the Reggay”** des Toots and the Maytals, p

In [ ]:
for p in search_knowledge("origin of reggae")["passages"]:
    print(p["source"], "->", p["text"][:400], "\n")

Reggae -> word we used to use in Jamaica called "streggae". If a girl is walking and the guys look at her and say "Man, she's streggae" it means she don't dress well, she look raggedy. The girls would say that about the men too. This one morning me and my two friends were playing and I said, "OK man, let's do the reggay." It was just something that came out of my mouth. So we just start singing "Do the regg 

Reggae -> Reggae ( ) is a music genre that originated in Jamaica in the late 1960s. The term also refers to the modern popular music of Jamaica and its diaspora. The 1968 single by Toots and the Maytals titled "Do the Reggay" was the first popular song to use the word reggae, effectively naming the genre and introducing it to a global audience. == Origins and influences == Reggae developed from earlier Jama 

Reggae -> guitar or piano chords on the offbeat. The bass guitar plays a central role, with a thick, heavy tone and the high frequencies reduced to accentuate the low end. Ca

In [ ]:
passages = search_knowledge("origin of reggae", n_results=3)["passages"]
full_text = " ".join(p["text"] for p in passages).lower()

for word in ["mento", "ska", "rocksteady", "calypso", "slower"]:
    print(f"{word:12} -> {'present' if word in full_text else 'ABSENT'}")


mento        -> present
ska          -> present
rocksteady   -> present
calypso      -> present
slower       -> present


In [ ]:
!pip install gradio -q


In [ ]:
import gradio as gr


def chat(question):
    answer, sources = ask_agent(question)
    if sources:
        lines = []
        for s in sources:
            if s.startswith("Base"):
                lines.append(f"- `{s}`")
            else:
                lines.append(f"- {s}")
        sources_md = "\n".join(lines)
    else:
        sources_md = "Aucune source utilisee"
    return f"{answer}\n\n---\n**Sources utilisees**\n\n{sources_md}"


demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Pose ta question", placeholder="Ex : D'ou vient le reggae ?"),
    outputs=gr.Markdown(),
    title="Assistant boutique de musique : RAG + base de donnees",
    description="Pose une question sur le catalogue et les ventes (base Chinook) ou sur l'histoire des genres musicaux (articles Wikipedia).",
    examples=[
        "Combien d'albums a le groupe AC/DC ?",
        "D'ou vient le reggae ?",
        "Quel genre est le plus represente dans la boutique, et que sait-on de ce genre ?"
    ],
    flagging_mode="never"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fe4cd8ffa0eecba871.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
